# ToolPort — Qwen3.5-4B Aggressive GGUF (AUTO Colab, Long Context)

Target utama: **Google Colab + NVIDIA T4 16 GB**. Notebook ini mengotomatisasi pemilihan quant, cache model, setup llama.cpp, fallback VRAM/context/KV, warm-up, benchmark, dan tunnel opsional.

Model: `HauhauCS/Qwen3.5-4B-Uncensored-HauhauCS-Aggressive`

> Jalankan dari atas ke bawah. Pada mode AUTO, notebook memilih konfigurasi berdasarkan VRAM yang tersedia.


In [ ]:
# 1) Dependencies — pinned agar kompatibel dengan Google Colab
!pip -q install -U "huggingface_hub>=0.34" psutil "requests==2.32.4"
!apt-get -qq update
!apt-get -qq install -y git cmake ninja-build build-essential curl
print("Dependencies ready.")


In [ ]:
# 2) User configuration
from pathlib import Path
import os, json, time, subprocess, shutil, csv

HF_REPO = "HauhauCS/Qwen3.5-4B-Uncensored-HauhauCS-Aggressive"
MODE = "AUTO"                 # AUTO or MANUAL
MANUAL_QUANT = "Q4_K_M"      # used only when MODE == MANUAL
ENABLE_VISION = True
ENABLE_EMBEDDED_MTP = False      # enable only after base server is proven stable
USE_GOOGLE_DRIVE_CACHE = True   # avoids re-downloading after runtime resets
ENABLE_TUNNEL = False           # enable only after local tests pass
PORT = 8080
HOST = "127.0.0.1"

API_KEY = None
try:
    from google.colab import userdata
    API_KEY = userdata.get("LLAMA_API_KEY")
except Exception:
    pass

ROOT = Path("/content/toolport")
LLAMA_DIR = ROOT / "llama.cpp"
LOG_DIR = ROOT / "logs"
RESULT_DIR = ROOT / "results"
for p in (ROOT, LOG_DIR, RESULT_DIR):
    p.mkdir(parents=True, exist_ok=True)

print("Mode       :", MODE)
print("Vision     :", ENABLE_VISION)
print("Drive cache:", USE_GOOGLE_DRIVE_CACHE)
print("API key    :", "configured" if API_KEY else "not configured")


In [ ]:
# 3) Detect GPU, choose quant automatically, prepare persistent cache
if not shutil.which("nvidia-smi"):
    raise RuntimeError("GPU tidak terdeteksi. Colab: Runtime > Change runtime type > T4 GPU.")

q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free,driver_version", "--format=csv,noheader,nounits"], text=True, capture_output=True, check=True).stdout.strip().split(',')
GPU_NAME = q[0].strip()
GPU_TOTAL_MIB = int(q[1].strip())
GPU_FREE_MIB = int(q[2].strip())
GPU_DRIVER = q[3].strip()
GPU_FREE_GIB = GPU_FREE_MIB / 1024

if MODE.upper() == "MANUAL":
    QUANT = MANUAL_QUANT
elif GPU_FREE_GIB >= 22:
    QUANT = "Q6_K"
else:
    # Q4_K_M is the long-context default for Colab T4: only ~2.6 GB of weights.
    QUANT = "Q4_K_M"

MODEL_FILE = f"Qwen3.5-4B-Uncensored-HauhauCS-Aggressive-{QUANT}.gguf"
MMPROJ_FILE = "mmproj-Qwen3.5-4B-Uncensored-HauhauCS-Aggressive-BF16.gguf"

if USE_GOOGLE_DRIVE_CACHE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        CACHE_ROOT = Path("/content/drive/MyDrive/ToolPort/cache")
        CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    except Exception as e:
        print("Drive cache unavailable; using local disk:", e)
        CACHE_ROOT = ROOT / "cache"
else:
    CACHE_ROOT = ROOT / "cache"

CACHE_MODEL_DIR = CACHE_ROOT / "models"
CACHE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_MODEL_DIR = ROOT / "models"
RUNTIME_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# llama.cpp already supports automatic GPU fitting; try it first.
GPU_LAYER_CANDIDATES = ["auto", 32, 24, 16]
# Qwen3.5-4B has 262K native context. Try largest first and degrade gracefully.
CONTEXT_CANDIDATES = [262144, 196608, 131072, 98304, 65536]
KV_CACHE_CANDIDATES = ["q8_0", "q4_0"]

print(f"GPU        : {GPU_NAME} | total={GPU_TOTAL_MIB/1024:.1f} GiB | free={GPU_FREE_GIB:.1f} GiB | driver={GPU_DRIVER}")
print("AUTO quant :", QUANT)
print("Model file :", MODEL_FILE)
print("Drive cache: ", CACHE_MODEL_DIR)
print("Runtime SSD: ", RUNTIME_MODEL_DIR)
print("NGL search :", GPU_LAYER_CANDIDATES)


In [ ]:
# 4) Download/reuse model cache
from huggingface_hub import hf_hub_download

def hf_download(filename):
    dst = CACHE_MODEL_DIR / filename
    if dst.exists() and dst.stat().st_size > 1024**3:
        print(f"CACHE HIT: {filename} ({dst.stat().st_size/1024**3:.2f} GiB)")
        return dst
    print("Downloading from Hugging Face:", filename)
    return Path(hf_hub_download(repo_id=HF_REPO, filename=filename, local_dir=str(CACHE_MODEL_DIR)))

def stage_local(src, chunk_mb=64):
    dst = RUNTIME_MODEL_DIR / src.name
    src_size = src.stat().st_size
    if dst.exists() and dst.stat().st_size == src_size:
        print("LOCAL SSD HIT:", dst)
        return dst

    free_bytes = shutil.disk_usage(RUNTIME_MODEL_DIR).free
    safety = 2 * 1024**3
    if free_bytes < src_size + safety:
        raise RuntimeError(
            f"Local SSD tidak cukup untuk staging {src.name}. "
            f"Need ~{(src_size+safety)/1024**3:.1f} GiB, free={free_bytes/1024**3:.1f} GiB."
        )

    if dst.exists():
        print("Removing incomplete local copy:", dst)
        dst.unlink()

    print(f"Staging {src.name} to local SSD ({src_size/1024**3:.2f} GiB)...")
    chunk = chunk_mb * 1024 * 1024
    copied = 0
    t0 = time.perf_counter()
    last_report = 0.0
    with src.open("rb", buffering=0) as fsrc, dst.open("wb", buffering=0) as fdst:
        while True:
            data = fsrc.read(chunk)
            if not data:
                break
            fdst.write(data)
            copied += len(data)
            now = time.perf_counter()
            if now - last_report >= 2 or copied == src_size:
                elapsed = max(now - t0, 0.001)
                speed = copied / elapsed / 1024**2
                pct = copied * 100 / src_size
                print(f"  {pct:5.1f}% | {copied/1024**3:.2f}/{src_size/1024**3:.2f} GiB | {speed:.1f} MiB/s")
                last_report = now

    if dst.stat().st_size != src_size:
        dst.unlink(missing_ok=True)
        raise RuntimeError("Local staging copy size mismatch; copy was incomplete.")
    print(f"LOCAL SSD READY: {dst}")
    return dst

download_t0 = time.perf_counter()
MODEL_PATH = stage_local(hf_download(MODEL_FILE))
MMPROJ_PATH = stage_local(hf_download(MMPROJ_FILE)) if ENABLE_VISION else None
DOWNLOAD_SEC = time.perf_counter() - download_t0
print(f"Model : {MODEL_PATH} ({MODEL_PATH.stat().st_size/1024**3:.2f} GiB)")
if MMPROJ_PATH:
    print(f"mmproj: {MMPROJ_PATH} ({MMPROJ_PATH.stat().st_size/1024**3:.2f} GiB)")
print(f"Download/cache stage: {DOWNLOAD_SEC:.1f}s")


In [ ]:
# 5) Fast llama.cpp setup: prebuilt first, source build only as fallback
import requests

def sh(args, cwd=None, check=True, **kwargs):
    print("+", " ".join(map(str, args)))
    return subprocess.run(args, cwd=cwd, check=check, **kwargs)

server_bin = None
SERVER_PREFIX = []
llama_app = Path.home() / ".llama-app" / "llama"
setup_t0 = time.perf_counter()
try:
    if not llama_app.exists():
        print("Installing official prebuilt llama.cpp...")
        installer = requests.get("https://llama.app/install.sh", timeout=60)
        installer.raise_for_status()
        sh(["sh"], input=installer.text, text=True, timeout=180)
    if llama_app.exists():
        probe = subprocess.run([str(llama_app), "version"], text=True, capture_output=True, timeout=20)
        if probe.returncode == 0:
            server_bin = llama_app
            SERVER_PREFIX = ["serve"]
            print("Using prebuilt:", probe.stdout.strip())
except Exception as e:
    print("Prebuilt unavailable; source-build fallback will be used:", type(e).__name__, e)

if server_bin is None:
    if not LLAMA_DIR.exists():
        sh(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp.git", str(LLAMA_DIR)])
    build_dir = LLAMA_DIR / "build"
    compiled_server = build_dir / "bin" / "llama-server"
    if not compiled_server.exists():
        cmake_args = ["cmake", "-S", str(LLAMA_DIR), "-B", str(build_dir), "-G", "Ninja", "-DGGML_CUDA=ON", "-DCMAKE_BUILD_TYPE=Release", "-DGGML_NATIVE=OFF"]
        if "T4" in GPU_NAME:
            cmake_args.append("-DCMAKE_CUDA_ARCHITECTURES=75")
        sh(cmake_args)
        jobs = max(2, os.cpu_count() or 2)
        sh(["cmake", "--build", str(build_dir), "--target", "llama-server", "-j", str(jobs)])
    server_bin = compiled_server

assert Path(server_bin).exists(), f"Server binary missing: {server_bin}"
help_cmd = [str(server_bin)] + SERVER_PREFIX + ["--help"]
help_text = subprocess.run(help_cmd, text=True, capture_output=True).stdout
SETUP_SEC = time.perf_counter() - setup_t0
print(f"Server runtime ready in {SETUP_SEC:.1f}s:", server_bin, *SERVER_PREFIX)
print("MTP support flag detected:", "--spec-type" in help_text)


In [ ]:
# 6) Adaptive launcher: VRAM-aware GPU offload + context + KV fallback
SERVER = None
ACTIVE_CONFIG = None
SERVER_LOG = LOG_DIR / "llama-server.log"

# Pick a free localhost port before launching. This avoids stale Colab processes
# from blocking 8080 and triggering pointless VRAM/context fallback attempts.
import socket

def port_is_free(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        try:
            s.bind((HOST, port))
            return True
        except OSError:
            return False

def choose_free_port(start_port, max_tries=50):
    for candidate in range(int(start_port), int(start_port) + max_tries):
        if port_is_free(candidate):
            return candidate
    raise RuntimeError(f"No free localhost port found in {start_port}..{int(start_port)+max_tries-1}")

REQUESTED_PORT = PORT
PORT = choose_free_port(PORT)
if PORT != REQUESTED_PORT:
    print(f"Port {REQUESTED_PORT} is already in use; automatically using port {PORT}.")
else:
    print(f"Using free port {PORT}.")

def supported(flag):
    return flag in help_text

def auth_headers():
    h = {"Content-Type": "application/json"}
    if API_KEY:
        h["Authorization"] = f"Bearer {API_KEY}"
    return h

def stop_server():
    global SERVER
    if SERVER is not None and SERVER.poll() is None:
        SERVER.terminate()
        try:
            SERVER.wait(timeout=6)
        except subprocess.TimeoutExpired:
            SERVER.kill()
    SERVER = None

def wait_http(timeout=180):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if SERVER.poll() is not None:
            return False
        try:
            # /health is the official readiness endpoint: 503 while loading, 200 when ready.
            r = requests.get(f"http://{HOST}:{PORT}/health", headers=auth_headers(), timeout=2)
            if r.status_code == 200:
                return True
        except requests.RequestException:
            pass
        time.sleep(1)
    return False

def smoke_test():
    payload = {"model": "local", "messages": [{"role": "user", "content": "Reply only with OK"}], "max_tokens": 4, "temperature": 0}
    r = requests.post(f"http://{HOST}:{PORT}/v1/chat/completions", headers=auth_headers(), json=payload, timeout=90)
    r.raise_for_status()
    return r.json()

def build_cmd(ctx, ngl, kv):
    cmd = [str(server_bin)] + SERVER_PREFIX + ["--model", str(MODEL_PATH), "--ctx-size", str(ctx), "--n-gpu-layers", str(ngl), "--parallel", "1", "--host", HOST, "--port", str(PORT)]
    optional_pairs = [("--batch-size", "512"), ("--ubatch-size", "128"), ("--cache-type-k", kv), ("--cache-type-v", kv), ("--flash-attn", "on"), ("--temp", "0.6"), ("--top-p", "0.95"), ("--top-k", "20")]
    for flag, value in optional_pairs:
        if supported(flag): cmd += [flag, value]
    if supported("--jinja"): cmd += ["--jinja"]
    if ENABLE_VISION and MMPROJ_PATH and supported("--mmproj"): cmd += ["--mmproj", str(MMPROJ_PATH)]
    if ENABLE_EMBEDDED_MTP and supported("--spec-type"): cmd += ["--spec-type", "draft-mtp"]
    if API_KEY and supported("--api-key"): cmd += ["--api-key", API_KEY]
    return cmd

attempts = []
launch_t0 = time.perf_counter()
stop_server()
for kv in KV_CACHE_CANDIDATES:
    if ACTIVE_CONFIG: break
    for ctx in CONTEXT_CANDIDATES:
        if ACTIVE_CONFIG: break
        for ngl in GPU_LAYER_CANDIDATES:
            stop_server()
            cmd = build_cmd(ctx, ngl, kv)
            print(f"\nTrying quant={QUANT} ctx={ctx} kv={kv} ngl={ngl}")
            safe = ["***" if API_KEY and x == API_KEY else x for x in cmd]
            print(" ".join(safe))
            log_f = open(SERVER_LOG, "w", buffering=1)
            SERVER = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT, text=True)
            if wait_http(180):
                try:
                    smoke_test()
                    ACTIVE_CONFIG = {"quant": QUANT, "context": ctx, "kv": kv, "gpu_layers": ngl}
                    print("PASS:", ACTIVE_CONFIG)
                    break
                except Exception as e:
                    err = f"smoke failed: {type(e).__name__}: {e}"
            else:
                err = "server did not become ready"
            attempts.append({"context": ctx, "kv": kv, "gpu_layers": ngl, "error": err})
            print("FAILED:", err)
            stop_server()
            try:
                log_f.close()
            except Exception:
                pass
            tail = SERVER_LOG.read_text(errors="ignore")[-2500:] if SERVER_LOG.exists() else ""
            if tail:
                print("--- server log tail ---")
                print(tail)
                print("--- end log ---")
            if "couldn't bind HTTP server socket" in tail:
                old_port = PORT
                PORT = choose_free_port(PORT + 1)
                print(f"Bind conflict detected on {old_port}; next attempt will use port {PORT}.")
            time.sleep(1)

LOAD_SEC = time.perf_counter() - launch_t0
if not ACTIVE_CONFIG:
    tail = SERVER_LOG.read_text(errors="ignore")[-10000:] if SERVER_LOG.exists() else ""
    raise RuntimeError("No stable configuration found.\n" + json.dumps(attempts, indent=2) + "\nLast server log:\n" + tail)
print(f"Stable server found in {LOAD_SEC:.1f}s")


In [ ]:
# 7) Warm-up + benchmark + optional vision self-test
import base64

def chat(messages, max_tokens=128, temperature=0.6):
    payload = {"model": "local", "messages": messages, "max_tokens": max_tokens, "temperature": temperature}
    t0 = time.perf_counter()
    r = requests.post(f"http://{HOST}:{PORT}/v1/chat/completions", headers=auth_headers(), json=payload, timeout=240)
    elapsed = time.perf_counter() - t0
    r.raise_for_status()
    return r.json(), elapsed

print("Warm-up...")
chat([{"role": "user", "content": "Say ready."}], max_tokens=8, temperature=0)
bench_res, BENCH_SEC = chat([{"role": "user", "content": "Explain GGUF in exactly three concise sentences."}], max_tokens=96)
BENCH_TEXT = bench_res["choices"][0]["message"].get("content", "")
usage = bench_res.get("usage", {})
COMPLETION_TOKENS = usage.get("completion_tokens") or 0
TOKENS_PER_SEC = COMPLETION_TOKENS / BENCH_SEC if COMPLETION_TOKENS else None
print("TEXT TEST: PASS")
print(BENCH_TEXT)
print(f"Elapsed: {BENCH_SEC:.2f}s")
if TOKENS_PER_SEC: print(f"Approx throughput: {TOKENS_PER_SEC:.2f} tok/s")

VISION_STATUS = "DISABLED"
if ENABLE_VISION and MMPROJ_PATH and supported("--mmproj"):
    # Tiny valid PNG embedded directly; avoids Pillow dependency issues in Colab.
    PNG_BASE64 = "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mNk+A8AAQUBAScY42YAAAAASUVORK5CYII="
    data_url = "data:image/png;base64," + PNG_BASE64
    msgs = [{"role": "user", "content": [{"type": "text", "text": "Describe this image briefly."}, {"type": "image_url", "image_url": {"url": data_url}}]}]
    try:
        vr, vs = chat(msgs, max_tokens=64)
        print("VISION TEST: PASS")
        print(vr["choices"][0]["message"].get("content", ""))
        VISION_STATUS = "PASS"
    except Exception as e:
        print("VISION TEST: FAILED", e)
        VISION_STATUS = "FAILED"

gpu_after = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.free,utilization.gpu", "--format=csv,noheader,nounits"], text=True, capture_output=True).stdout.strip()
print("GPU after benchmark:", gpu_after)


In [ ]:
# 8) Save research/run report (JSON + CSV)
from datetime import datetime, timezone

REPORT = {"timestamp_utc": datetime.now(timezone.utc).isoformat(), "gpu": GPU_NAME, "gpu_total_mib": GPU_TOTAL_MIB, "gpu_free_before_mib": GPU_FREE_MIB, "driver": GPU_DRIVER, "repo": HF_REPO, "model_file": MODEL_FILE, "quant": QUANT, "vision": ENABLE_VISION, "vision_test": VISION_STATUS, "server_port": PORT, "context": ACTIVE_CONFIG["context"], "kv_cache": ACTIVE_CONFIG["kv"], "gpu_layers": ACTIVE_CONFIG["gpu_layers"], "download_stage_sec": round(DOWNLOAD_SEC, 2), "runtime_setup_sec": round(SETUP_SEC, 2), "server_search_sec": round(LOAD_SEC, 2), "benchmark_sec": round(BENCH_SEC, 2), "completion_tokens": COMPLETION_TOKENS, "approx_tokens_per_sec": round(TOKENS_PER_SEC, 3) if TOKENS_PER_SEC else None, "gpu_after": gpu_after}
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
json_path = RESULT_DIR / f"run_{stamp}.json"
json_path.write_text(json.dumps(REPORT, indent=2), encoding="utf-8")
csv_path = RESULT_DIR / "runs.csv"
write_header = not csv_path.exists()
with csv_path.open("a", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(REPORT.keys()))
    if write_header: w.writeheader()
    w.writerow(REPORT)
print("\n================ AiLab Runtime Report ================")
print(f"GPU          : {GPU_NAME} ({GPU_TOTAL_MIB/1024:.1f} GiB)")
print(f"Quant        : {QUANT}")
print(f"Vision       : {ENABLE_VISION} / test={VISION_STATUS}")
print(f"Context      : {ACTIVE_CONFIG['context']}")
print(f"GPU layers   : {ACTIVE_CONFIG['gpu_layers']}")
print(f"KV cache     : {ACTIVE_CONFIG['kv']}")
print(f"Server port  : {PORT}")
print(f"Generation   : {TOKENS_PER_SEC:.2f} tok/s" if TOKENS_PER_SEC else "Generation   : n/a")
print("JSON report  :", json_path)
print("CSV history  :", csv_path)
print("=======================================================")


In [ ]:
# 9) Optional Cloudflare Quick Tunnel — auto-detect the active llama.cpp port
import re, psutil
TUNNEL = None
PUBLIC_URL = None

def probe_llama_port(port):
    try:
        health = requests.get(f"http://{HOST}:{port}/health", headers=auth_headers(), timeout=2)
        if health.status_code != 200:
            return False
        models = requests.get(f"http://{HOST}:{port}/v1/models", headers=auth_headers(), timeout=3)
        return models.status_code == 200
    except requests.RequestException:
        return False

def detect_active_llama_port():
    candidates = []

    # First choice: read the port from the exact subprocess started by this notebook.
    if "SERVER" in globals() and SERVER is not None and SERVER.poll() is None:
        try:
            args = list(SERVER.args)
            if "--port" in args:
                candidates.append(int(args[args.index("--port") + 1]))
        except Exception:
            pass

    # Second choice: the notebook's latest PORT value.
    if "PORT" in globals():
        try:
            candidates.append(int(PORT))
        except Exception:
            pass

    # Final fallback: inspect every local listening port and verify llama.cpp endpoints.
    try:
        for conn in psutil.net_connections(kind="inet"):
            if conn.status != psutil.CONN_LISTEN or not conn.laddr:
                continue
            bind_ip = getattr(conn.laddr, "ip", conn.laddr[0])
            bind_port = getattr(conn.laddr, "port", conn.laddr[1])
            if bind_ip in ("127.0.0.1", "0.0.0.0", "::1", "::"):
                candidates.append(int(bind_port))
    except Exception as e:
        print("Listening-port scan unavailable:", e)

    seen = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if probe_llama_port(candidate):
            return candidate

    raise RuntimeError("No active llama.cpp server detected on localhost. Run the server cell first.")

if not ENABLE_TUNNEL:
    print("Tunnel disabled. Set ENABLE_TUNNEL=True only after local tests pass.")
else:
    ACTIVE_TUNNEL_PORT = detect_active_llama_port()
    PORT = ACTIVE_TUNNEL_PORT
    print(f"Active llama.cpp server detected on {HOST}:{ACTIVE_TUNNEL_PORT}")
    if not API_KEY:
        print("WARNING: LLAMA_API_KEY is not configured; endpoint may be public without authentication.")
    cf = Path("/content/cloudflared")
    if not cf.exists():
        r = requests.get("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", timeout=120)
        r.raise_for_status(); cf.write_bytes(r.content); cf.chmod(0o755)
    tunnel_log = LOG_DIR / "cloudflared.log"
    lf = open(tunnel_log, "w", buffering=1)
    TUNNEL = subprocess.Popen([str(cf), "tunnel", "--url", f"http://{HOST}:{PORT}", "--no-autoupdate"], stdout=lf, stderr=subprocess.STDOUT, text=True)
    deadline = time.time() + 45
    last_notice = -1
    while time.time() < deadline:
        elapsed = int(45 - (deadline - time.time()))
        if elapsed // 5 != last_notice:
            last_notice = elapsed // 5; print(f"Waiting for tunnel... {elapsed}s")
        text = tunnel_log.read_text(errors="ignore") if tunnel_log.exists() else ""
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
        if m:
            PUBLIC_URL = m.group(0); break
        if TUNNEL.poll() is not None: break
        time.sleep(1)
    if PUBLIC_URL:
        print("Public URL :", PUBLIC_URL)
        print("OpenAI base:", PUBLIC_URL + "/v1")
        print("Auth       :", "Bearer LLAMA_API_KEY" if API_KEY else "NO API KEY")
    else:
        text = tunnel_log.read_text(errors="ignore")[-6000:] if tunnel_log.exists() else ""
        raise RuntimeError("Cloudflare tunnel did not produce a URL within 45s.\n" + text)


## Cara menjalankan

1. Colab: **Runtime → Change runtime type → T4 GPU**.
2. Run All. Jika Drive cache aktif, izinkan mount sekali.
3. Run pertama mengunduh model; run berikutnya memakai cache Drive bila file tersedia.
4. Biarkan `ENABLE_TUNNEL=False` sampai benchmark lokal berhasil.
5. Setelah stabil, ubah `ENABLE_TUNNEL=True` dan jalankan cell tunnel saja.

Mode AUTO memakai Q4_K_M pada T4 untuk memaksimalkan ruang long-context. Launcher mencoba 262K lalu turun bertahap sampai menemukan konfigurasi stabil.
